# 01 — Benchmark tagging + subset report

Tags both test datasets for conflict/canary/grammar/question-type subsets. No model inference, no correction.

In [ ]:

import json, re, os, math, csv, statistics
from pathlib import Path
from collections import Counter, defaultdict

CANDIDATE_DIRS = [Path('/kaggle/working'), Path('/kaggle/input/datasets/yixuanisthebest/v2333333'), Path('/kaggle/input/datasets/nguyenminhtric/test-pipeline'), Path('/mnt/data')]

def find_file(names, required=True):
    if isinstance(names, str): names=[names]
    for name in names:
        p=Path(name)
        if p.exists(): return p
    for d in CANDIDATE_DIRS:
        if not d.exists(): continue
        for name in names:
            p=d/name
            if p.exists(): return p
    if required:
        raise FileNotFoundError(f'Could not find any of {names} in {CANDIDATE_DIRS}')
    return None

def load_json(names, required=True):
    p=find_file(names, required=required)
    if p is None: return None, None
    with open(p,'r',encoding='utf-8') as f: return json.load(f), p

def save_json(obj, path):
    path=Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path,'w',encoding='utf-8') as f: json.dump(obj,f,ensure_ascii=False,indent=2)
    # reload verify
    with open(path,'r',encoding='utf-8') as f: json.load(f)
    print('saved', path)

def norm_text(s): return re.sub(r'\s+',' ',str(s).strip())
def norm_key(s): return re.sub(r'[^a-z0-9]+',' ',str(s).lower()).strip()

def camel_to_words(s):
    s=re.sub(r'([a-z])([A-Z])', r'\1 \2', s)
    return norm_key(s)

def label_norm(x):
    if x is None: return None
    s=str(x).strip()
    if s.upper() in ['A','B','C','D']: return s.upper()
    t=s.title()
    return t if t in ['Yes','No','Unknown','Unparseable'] else s

def is_mc_question(q):
    return bool(re.search(r'\n\s*A\.', str(q))) or all(x in str(q) for x in ['A.','B.'])

def flatten_dataset(rows, dataset_name='dataset'):
    flat=[]
    for ri,row in enumerate(rows):
        qs=row.get('questions') if isinstance(row,dict) else None
        ans=row.get('answers') if isinstance(row,dict) else None
        exps=row.get('explanation') if isinstance(row,dict) else None
        if isinstance(qs, list):
            for qi,q in enumerate(qs):
                flat.append({
                    'flat_id': f'{dataset_name}_{ri}_{qi}',
                    'row_i': ri, 'q_i': qi,
                    'question': q,
                    'gold': label_norm(ans[qi]) if isinstance(ans,list) and qi < len(ans) else None,
                    'explanation': exps[qi] if isinstance(exps,list) and qi < len(exps) else None,
                    'premises_FOL': row.get('premises-FOL') or row.get('premises_FOL') or [],
                    'premises_NL': row.get('premises-NL') or row.get('premises_NL') or [],
                    'idx': row.get('idx'),
                    'is_mc': is_mc_question(q),
                    'is_ynu': not is_mc_question(q),
                })
        elif isinstance(row, dict):
            q=row.get('question') or row.get('query') or ''
            flat.append({
                'flat_id': f'{dataset_name}_{ri}_0', 'row_i': ri, 'q_i': 0,
                'question': q, 'gold': label_norm(row.get('answer') or row.get('gold')),
                'explanation': row.get('explanation'),
                'premises_FOL': row.get('premises-FOL') or row.get('premises_FOL') or [],
                'premises_NL': row.get('premises-NL') or row.get('premises_NL') or [],
                'idx': row.get('idx'), 'is_mc': is_mc_question(q), 'is_ynu': not is_mc_question(q)
            })
    return flat

ATOM_RE = re.compile(r'¬?([A-Z][A-Za-z0-9_]*)\(x\)')

def parse_fol_items(fols):
    facts_pos=set(); facts_neg=set(); exists_pos=set(); exists_neg=set(); impl=[]; raw=[]
    for s in fols or []:
        t=str(s).replace('→','->').replace('∀','forall').replace('∃','exists').replace('¬','~')
        raw.append(str(s))
        # exists conjunctions
        if 'exists' in t:
            for m in re.finditer(r'(~?)([A-Z][A-Za-z0-9_]*)\(x\)', t):
                (exists_neg if m.group(1) else exists_pos).add(m.group(2))
        elif 'forall' in t and '->' not in t:
            m=re.search(r'forall\s*x\s*\(?\s*(~?)([A-Z][A-Za-z0-9_]*)\(x\)', t)
            if m:
                (facts_neg if m.group(1) else facts_pos).add(m.group(2))
        if 'forall' in t and '->' in t:
            parts=t.split('->',1)
            left=parts[0]; right=parts[1]
            ml=re.findall(r'(~?)([A-Z][A-Za-z0-9_]*)\(x\)', left)
            mr=re.findall(r'(~?)([A-Z][A-Za-z0-9_]*)\(x\)', right)
            if ml and mr:
                a_neg,a=ml[-1]; b_neg,b=mr[0]
                impl.append((a, bool(a_neg), b, bool(b_neg), str(s)))
    return {'facts_pos':facts_pos,'facts_neg':facts_neg,'exists_pos':exists_pos,'exists_neg':exists_neg,'impl':impl,'raw':raw}

def closure(parsed):
    pos=set(parsed['facts_pos']); neg=set(parsed['facts_neg']); paths={('pos',p):f'GIVEN ∀x {p}' for p in pos}; paths.update({('neg',p):f'GIVEN ∀x ¬{p}' for p in neg})
    changed=True
    while changed:
        changed=False
        for a,an,b,bn,raw in parsed['impl']:
            # forward
            if not an and a in pos:
                target=neg if bn else pos; key=('neg' if bn else 'pos', b)
                if b not in target:
                    target.add(b); paths[key]=raw; changed=True
            if an and a in neg:
                target=neg if bn else pos; key=('neg' if bn else 'pos', b)
                if b not in target:
                    target.add(b); paths[key]=raw; changed=True
            # contrapositive for single-antecedent implication: A -> B gives ~B -> ~A
            if not an and not bn and b in neg and a not in neg:
                neg.add(a); paths[('neg',a)]='CONTRAPOSITIVE_OF: '+raw; changed=True
            if not an and bn and b in pos and a not in neg:
                neg.add(a); paths[('neg',a)]='CONTRAPOSITIVE_OF: '+raw; changed=True
    return pos,neg,paths

def target_from_question(q):
    # approximate by matching CamelCase words against question tokens; fallback none
    return None

def predicate_candidates(fols):
    preds=set()
    for s in fols or []:
        preds.update(re.findall(r'([A-Z][A-Za-z0-9_]*)\(x\)', str(s)))
    return sorted(preds)

def best_predicate_match(text, predicates):
    txt=norm_key(text)
    best=(0,None)
    for p in predicates:
        words=camel_to_words(p).split()
        if not words: continue
        hit=sum(1 for w in words if w in txt)
        score=hit/len(words)
        # exact full phrase bonus
        if camel_to_words(p) in txt: score=max(score,1.0)
        if score>best[0]: best=(score,p)
    return best[1], best[0]

def qtype(q):
    s=str(q).lower()
    if is_mc_question(q): return 'mc'
    if 'at least one' in s or 'there is at least one' in s or re.search(r'\bsome\b', s): return 'existential'
    if re.search(r'\b(all|every)\b', s): return 'universal'
    if 'statement:' in s and 'if ' in s and ' then ' in s: return 'statement_conditional'
    if 'no ' in s: return 'universal_negative'
    return 'other_ynu'

def answer_from_explanation(exp):
    if not exp: return None
    m=re.findall(r'(?:answer is|final answer\s*[:\-]?)\s*(Yes|No|Unknown|[ABCD])\b', str(exp), flags=re.I)
    return label_norm(m[-1]) if m else None


In [ ]:

TEST_FILES = {
    'benchmark_v2_1000': 'benchmark_v2_1000.json',
    'generated_v4style_300': 'generated_v4style_300.json',
}
OUT_DIR=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')


In [ ]:

def detect_direct_conflict(fols):
    p=parse_fol_items(fols)
    pos,neg,paths=closure(p)
    direct=[]
    for pred in sorted((p['exists_pos'] | pos) & neg):
        direct.append({'predicate':pred,'type':'universal_negative_vs_positive_or_existential'})
    for pred in sorted(p['exists_pos'] & p['facts_neg']):
        if not any(d['predicate']==pred for d in direct):
            direct.append({'predicate':pred,'type':'forall_not_vs_exists'})
    return direct

def grammar_flags(flat):
    text=' '.join([str(x) for x in (flat.get('premises_NL') or [])])+' '+str(flat.get('question',''))
    flags=[]
    if re.search(r'\ba (intern|athlete|exam|employee|applicant|advisor|analyst|engineer)\b', text, flags=re.I): flags.append('ARTICLE_A_AN')
    q=str(flat.get('question',''))
    if re.search(r'\bDo all [a-z]+s? [a-z]+\?$', q.strip(), flags=re.I): flags.append('UNNATURAL_QUESTION_GRAMMAR')
    if re.search(r'\bdoes at least one [a-z]+ [a-z]+s\b', q, flags=re.I): flags.append('VERB_AGREEMENT_ODD')
    return flags

def tag_flat(flat):
    conflicts=detect_direct_conflict(flat.get('premises_FOL') or [])
    qt=qtype(flat['question'])
    exp_ans=answer_from_explanation(flat.get('explanation'))
    ans_mismatch = exp_ans is not None and flat.get('gold') is not None and exp_ans != flat.get('gold')
    preds=predicate_candidates(flat.get('premises_FOL') or [])
    target,score=best_predicate_match(flat.get('question',''), preds)
    expected=[]
    if qt=='existential' and flat.get('gold')=='No': expected.append('E1_FORALL_NEG_NOT_EXISTS')
    if qt=='universal' and flat.get('gold')=='Yes': expected.append('UNIVERSAL_FORWARD_CHAIN')
    if flat.get('is_mc'): expected.append('MC_OPTION_PROOF_ANALYSIS')
    if flat.get('gold')=='Unknown': expected.append('UNKNOWN_UNSUPPORTED_OR_AMBIGUOUS')
    return {**flat,
        'question_type': qt,
        'target_predicate_guess': target,
        'target_score': score,
        'answer_in_explanation': exp_ans,
        'answer_explanation_mismatch': ans_mismatch,
        'direct_fol_conflicts': conflicts,
        'has_direct_fol_conflict': bool(conflicts),
        'grammar_flags': grammar_flags(flat),
        'is_grammar_noisy': bool(grammar_flags(flat)),
        'expected_verifier_rules': expected,
    }

combined={}
for name,fn in TEST_FILES.items():
    rows,path=load_json(fn, required=True)
    flat=flatten_dataset(rows, name)
    tagged=[tag_flat(x) for x in flat]
    # row-level summaries
    summary={
        'dataset': name, 'input_path': str(path), 'raw_records': len(rows), 'flattened_samples': len(flat),
        'label_distribution': dict(Counter(x.get('gold') for x in tagged)),
        'question_type_distribution': dict(Counter(x.get('question_type') for x in tagged)),
        'answer_explanation_mismatch_samples': sum(x['answer_explanation_mismatch'] for x in tagged),
        'direct_fol_conflict_samples': sum(x['has_direct_fol_conflict'] for x in tagged),
        'grammar_noisy_samples': sum(x['is_grammar_noisy'] for x in tagged),
        'mc_samples': sum(x['is_mc'] for x in tagged), 'ynu_samples': sum(x['is_ynu'] for x in tagged),
    }
    save_json(tagged, OUT_DIR/f'{name}_tagged_flat.json')
    save_json(summary, OUT_DIR/f'{name}_subset_report.json')
    # CSV compact
    csv_path=OUT_DIR/f'{name}_tagged_flat.csv'
    with open(csv_path,'w',encoding='utf-8',newline='') as f:
        fields=['flat_id','row_i','q_i','gold','is_mc','question_type','target_predicate_guess','target_score','answer_explanation_mismatch','has_direct_fol_conflict','grammar_flags','expected_verifier_rules','question']
        w=csv.DictWriter(f, fieldnames=fields); w.writeheader()
        for x in tagged:
            y={k:x.get(k) for k in fields}; y['grammar_flags']='|'.join(x.get('grammar_flags') or []); y['expected_verifier_rules']='|'.join(x.get('expected_verifier_rules') or [])
            w.writerow(y)
    print('\n', name, json.dumps(summary, ensure_ascii=False, indent=2)[:2000])
    combined[name]=summary
save_json(combined, OUT_DIR/'01_combined_subset_report.json')
